In [1]:
import os
os.makedirs("TP6_outputs", exist_ok=True)

# ============================================
# 1. LOAD DATA
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.api import VAR
from scipy.stats import chi2

data = pd.read_excel("SP_WTI_data.xlsx", index_col=0)
data.index = pd.to_datetime(data.index)

# rename for clarity
data.columns = ["SP500", "WTI"]

In [2]:
# ============================================
# 2. RETURNS
# ============================================

returns = np.log(data).diff().dropna()

In [3]:
# ============================================
# 3. ROLLING VARIANCE (60 days)
# ============================================

window = 60

var_sp = returns["SP500"].rolling(window).var()
var_wti = returns["WTI"].rolling(window).var()

df = pd.concat([var_sp, var_wti], axis=1).dropna()
df.columns = ["SP500_var", "WTI_var"]

In [4]:
# ============================================
# 4. VAR ESTIMATION (FULL SAMPLE)
# ============================================

model = VAR(df)
lag = 2   # simple choice consistent with lecture
results = model.fit(lag)

residuals = results.resid.values
T = residuals.shape[0]

Sigma = np.cov(residuals.T)

C:\Users\rapha\PyCharmMiscProject\Project-IMF\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [5]:
# ============================================
# 5. AUXILIARY MODELS (NO CROSS TERMS)
# ============================================

# SP500 equation without WTI
Y_sp = df["SP500_var"].values
X_sp = []

for i in range(1, lag+1):
    X_sp.append(df["SP500_var"].shift(i).values)

X_sp = np.column_stack(X_sp)
X_sp = X_sp[lag:]
Y_sp = Y_sp[lag:]

beta_sp = np.linalg.inv(X_sp.T @ X_sp) @ (X_sp.T @ Y_sp)
res_sp = Y_sp - X_sp @ beta_sp
Sigma1 = np.var(res_sp)


# WTI equation without SP500
Y_wti = df["WTI_var"].values
X_wti = []

for i in range(1, lag+1):
    X_wti.append(df["WTI_var"].shift(i).values)

X_wti = np.column_stack(X_wti)
X_wti = X_wti[lag:]
Y_wti = Y_wti[lag:]

beta_wti = np.linalg.inv(X_wti.T @ X_wti) @ (X_wti.T @ Y_wti)
res_wti = Y_wti - X_wti @ beta_wti
Sigma2 = np.var(res_wti)

In [6]:
# ============================================
# 6. RESTRICTED SYSTEM (NO INSTANTANEOUS LINK)
# ============================================

Sigma_diag = np.diag(np.diag(Sigma))

In [7]:
# ============================================
# 7. GEWEKE CAUSALITY MEASURES
# ============================================

# determinants
det_Sigma = np.linalg.det(Sigma)
det_Sigma_diag = np.linalg.det(Sigma_diag)

# convert scalars to matrix form
Sigma1_mat = np.array([[Sigma1]])
Sigma2_mat = np.array([[Sigma2]])

# extract blocks
Sigma11 = Sigma[0,0]
Sigma22 = Sigma[1,1]

# --- measures ---

# WTI → SP500
C_wti_sp = np.log(Sigma1 / Sigma11)

# SP500 → WTI
C_sp_wti = np.log(Sigma2 / Sigma22)

# instantaneous
C_inst = np.log((Sigma11 * Sigma22) / det_Sigma)

# total
C_total = C_wti_sp + C_sp_wti + C_inst

In [8]:
# ============================================
# 8. TEST STATISTICS (CHI2)
# ============================================

df_gc = lag

stat_wti_sp = T * C_wti_sp
pval_wti_sp = 1 - chi2.cdf(stat_wti_sp, df_gc)

stat_sp_wti = T * C_sp_wti
pval_sp_wti = 1 - chi2.cdf(stat_sp_wti, df_gc)

stat_inst = T * C_inst
pval_inst = 1 - chi2.cdf(stat_inst, 1)

stat_total = T * C_total
pval_total = 1 - chi2.cdf(stat_total, 2*lag + 1)

In [9]:
# ============================================
# 9. RESULTS TABLE
# ============================================

results_full = pd.DataFrame({
    "Test": ["WTI → SP500", "SP500 → WTI", "Instantaneous", "Total"],
    "Statistic": [stat_wti_sp, stat_sp_wti, stat_inst, stat_total],
    "p-value": [pval_wti_sp, pval_sp_wti, pval_inst, pval_total]
})

print(results_full)

results_full.to_latex("TP6_outputs/full_sample.tex", index=False)

            Test   Statistic       p-value
0    WTI → SP500   70.484292  4.440892e-16
1    SP500 → WTI  100.417637  0.000000e+00
2  Instantaneous  114.484010  0.000000e+00
3          Total  285.385939  0.000000e+00


In [10]:
# ============================================
# 10. SUBSAMPLES (5-YEAR BLOCKS)
# ============================================

years = df.index.year
start_year = years.min()
end_year = years.max()

blocks = []
current = start_year

while current + 4 <= end_year:
    
    sub = df[(years >= current) & (years <= current + 4)]
    
    if len(sub) < 200:
        current += 5
        continue
    
    model = VAR(sub)
    results = model.fit(lag)
    
    residuals = results.resid.values
    T = residuals.shape[0]
    
    Sigma = np.cov(residuals.T)
    
    Sigma11 = Sigma[0,0]
    Sigma22 = Sigma[1,1]
    det_Sigma = np.linalg.det(Sigma)
    
    # simple approximations (same logic as full sample)
    C_wti_sp = np.log(Sigma1 / Sigma11)
    C_sp_wti = np.log(Sigma2 / Sigma22)
    C_inst = np.log((Sigma11 * Sigma22) / det_Sigma)
    C_total = C_wti_sp + C_sp_wti + C_inst
    
    stat = T * C_wti_sp
    pval = 1 - chi2.cdf(stat, lag)
    
    blocks.append([f"{current}-{current+4}", stat, pval])
    
    current += 5

subsample = pd.DataFrame(blocks, columns=["Period", "Stat", "p-value"])
print(subsample)

subsample.to_latex("TP6_outputs/subsample.tex", index=False)

      Period         Stat  p-value
0  2005-2009  -861.241362      1.0
1  2010-2014  1941.182262      0.0
2  2015-2019  2910.520827      0.0
3  2020-2024  -432.389797      1.0


C:\Users\rapha\PyCharmMiscProject\Project-IMF\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\rapha\PyCharmMiscProject\Project-IMF\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\rapha\PyCharmMiscProject\Project-IMF\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\rapha\PyCharmMiscProject\Project-IMF\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but i

In [11]:
# ============================================
# 11. FIGURE
# ============================================

plt.figure()
plt.plot(df.index, df["SP500_var"])
plt.plot(df.index, df["WTI_var"])
plt.title("Rolling Variances")
plt.savefig("TP6_outputs/variance.jpg")
plt.close()